# **Semana 13: Docker y Alta Disponibilidad (NB1 - Conceptual)**

## **Propósito de la Sesión**
Aprender a desplegar bases de datos en contenedores Docker y comprender los fundamentos de alta disponibilidad. Utilizaremos **Play with Docker** (entorno gratuito en línea) para levantar contenedores PostgreSQL, conectarnos desde otros contenedores, simular fallos y verificar la persistencia de datos.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Explicar** por qué los contenedores son útiles para desplegar bases de datos.
2. **Crear** y ejecutar contenedores PostgreSQL usando Docker.
3. **Configurar** volúmenes para persistencia de datos.
4. **Conectar** múltiples contenedores mediante redes Docker.
5. **Simular** fallos y reinicios para comprobar la resiliencia.
6. **Comprender** los conceptos básicos de alta disponibilidad (réplicas, failover).

## **Configuración Inicial: Play with Docker**

### **¿Qué es Play with Docker?**

Play with Docker (PWD) es un entorno Docker gratuito en el navegador. Proporciona una terminal Linux con Docker preinstalado y acceso temporal (4 horas). Ideal para prácticas sin instalar nada localmente.

### **Pasos para acceder:**

1. Abre [https://labs.play-with-docker.com/](https://labs.play-with-docker.com/) en tu navegador.
2. Inicia sesión con tu cuenta de Docker Hub (puedes crear una gratis si no tienes).
3. Haz clic en **"Start"** para comenzar una sesión.
4. En el panel izquierdo, haz clic en **"Add new instance"** para crear una instancia (nodo) con terminal.

✅ Ya tienes un terminal Docker listo para usar.

**Nota:** Este notebook contiene instrucciones y comandos. Puedes copiarlos y pegarlos directamente en la terminal de PWD.

---
## **Parte 1: Conceptos Fundamentales de Docker para Bases de Datos**

### **¿Por qué usar contenedores para bases de datos?**

*   **Consistencia**: Mismo entorno en desarrollo, pruebas y producción.
*   **Aislamiento**: Cada base de datos corre en su propio contenedor.
*   **Portabilidad**: Fácil mover entre máquinas/clouds.
*   **Rápido aprovisionamiento**: Levantar una BD en segundos.
*   **Versionado**: Control de versiones de la imagen (postgres:15, mysql:8, etc.)

### **Componentes clave:**

*   **Imagen**: Plantilla con el sistema operativo y el software (ej. postgres:15).
*   **Contenedor**: Instancia en ejecución de una imagen.
*   **Volumen**: Almacenamiento persistente fuera del contenedor.
*   **Red**: Comunicación entre contenedores.

---
## **Parte 2: Levantar un Contenedor PostgreSQL**

### **2.1. Descargar la imagen de PostgreSQL**

In [ ]:
# En la terminal de Play with Docker, ejecuta:
docker pull postgres:15

# Verificar imágenes descargadas
docker images

### **2.2. Ejecutar el contenedor PostgreSQL**

Ejecutaremos un contenedor con configuración básica:

In [ ]:
docker run -d \
  --name postgres1 \
  -e POSTGRES_PASSWORD=admin123 \
  -e POSTGRES_USER=admin \
  -e POSTGRES_DB=testdb \
  -p 5432:5432 \
  postgres:15

# Explicación:
# -d: ejecutar en segundo plano
# --name: nombre del contenedor
# -e: variables de entorno (configuran usuario, contraseña, BD inicial)
# -p: mapeo de puertos (host:contenedor)

# Verificar que el contenedor está corriendo
docker ps

### **2.3. Conectarse a PostgreSQL desde el host**

Podemos usar el cliente `psql` incluido en el contenedor para conectarnos a la base de datos.

In [ ]:
# Ejecutar psql dentro del contenedor
docker exec -it postgres1 psql -U admin -d testdb

# Una vez dentro de psql, ejecuta estos comandos:
CREATE TABLE personas (id SERIAL PRIMARY KEY, nombre VARCHAR(100));
INSERT INTO personas (nombre) VALUES ('Ana'), ('Carlos'), ('María');
SELECT * FROM personas;
\q

---
## **Parte 3: Persistencia de Datos con Volúmenes**

**Problema:** Si eliminamos el contenedor, los datos se pierden.

**Solución:** Usar volúmenes para persistir datos fuera del contenedor.

### **3.1. Crear un volumen y ejecutar contenedor con persistencia**

In [ ]:
# Crear un volumen (almacenamiento persistente gestionado por Docker)
docker volume create pgdata

# Listar volúmenes
docker volume ls

# Ejecutar nuevo contenedor con el volumen
docker run -d \
  --name postgres-persistente \
  -e POSTGRES_PASSWORD=admin123 \
  -e POSTGRES_USER=admin \
  -e POSTGRES_DB=testdb \
  -v pgdata:/var/lib/postgresql/data \
  -p 5433:5432 \
  postgres:15

# Nota: Usamos puerto 5433 para no conflictuar con el contenedor anterior

# Conectar y crear datos
docker exec -it postgres-persistente psql -U admin -d testdb -c "CREATE TABLE productos (id SERIAL PRIMARY KEY, nombre TEXT);"
docker exec -it postgres-persistente psql -U admin -d testdb -c "INSERT INTO productos (nombre) VALUES ('Laptop'), ('Mouse');"
docker exec -it postgres-persistente psql -U admin -d testdb -c "SELECT * FROM productos;"

### **3.2. Simular fallo y recuperación**

Eliminaremos el contenedor y crearemos uno nuevo usando el mismo volumen para comprobar que los datos persisten.

In [ ]:
# Detener y eliminar el contenedor
docker stop postgres-persistente
docker rm postgres-persistente

# Verificar que el volumen aún existe
docker volume ls

# Crear NUEVO contenedor con el MISMO volumen
docker run -d \
  --name postgres-nuevo \
  -e POSTGRES_PASSWORD=admin123 \
  -e POSTGRES_USER=admin \
  -e POSTGRES_DB=testdb \
  -v pgdata:/var/lib/postgresql/data \
  -p 5434:5432 \
  postgres:15

# Verificar que los datos persisten
docker exec -it postgres-nuevo psql -U admin -d testdb -c "SELECT * FROM productos;"

# Deberías ver los productos insertados anteriormente

✅ **Conclusión:** Los volúmenes permiten que los datos sobrevivan al ciclo de vida del contenedor.

---
## **Parte 4: Redes en Docker - Conexión entre Contenedores**

En aplicaciones reales, la base de datos y la aplicación suelen estar en contenedores separados. Necesitan comunicarse a través de una red.

### **4.1. Crear una red personalizada**

In [ ]:
# Crear red llamada 'app-network'
docker network create app-network

# Listar redes
docker network ls

### **4.2. Ejecutar contenedor PostgreSQL en la red**

In [ ]:
# Ejecutar PostgreSQL en la red app-network
docker run -d \
  --name postgres-red \
  --network app-network \
  -e POSTGRES_PASSWORD=admin123 \
  -e POSTGRES_USER=admin \
  -e POSTGRES_DB=testdb \
  -v pgdata2:/var/lib/postgresql/data \
  postgres:15

# Nota: No exponemos puertos porque la comunicación será interna

### **4.3. Ejecutar otro contenedor (cliente) en la misma red**

Usaremos un contenedor con `psql` para conectarnos a la base de datos.

In [ ]:
# Ejecutar contenedor cliente (basado en la misma imagen postgres)
docker run -it --rm \
  --name psql-client \
  --network app-network \
  postgres:15 psql -h postgres-red -U admin -d testdb

# Cuando pida contraseña, ingresa: admin123

# Una vez conectado, prueba:
SELECT * FROM productos;  -- (si no existe, crea alguna tabla)
CREATE TABLE ejemplos (id SERIAL, dato TEXT);
INSERT INTO ejemplos (dato) VALUES ('prueba red');
SELECT * FROM ejemplos;
\q

✅ **Importante:** Los contenedores en la misma red pueden comunicarse usando el nombre del contenedor como hostname (resolución DNS automática de Docker).

---
## **Parte 5: Simulación de Fallo y Reinicio**

Veremos qué sucede cuando un contenedor falla y cómo Docker puede reiniciarlo automáticamente.

### **5.1. Ejecutar contenedor con política de reinicio**

In [ ]:
# Ejecutar PostgreSQL con política de reinicio: --restart always
docker run -d \
  --name postgres-ha \
  --restart always \
  -e POSTGRES_PASSWORD=admin123 \
  -e POSTGRES_USER=admin \
  -e POSTGRES_DB=testdb \
  -v pgdata-ha:/var/lib/postgresql/data \
  -p 5435:5432 \
  postgres:15

# Verificar que está corriendo
docker ps

### **5.2. Simular fallo matando el proceso principal**

In [ ]:
# Obtener el PID del proceso postgres dentro del contenedor
docker top postgres-ha

# Matar el proceso (simular crash)
# (Necesitas el PID de la salida anterior, normalmente el primer número)
docker exec postgres-ha kill -9 1  # Esto mata el proceso principal

# Observar que el contenedor se reinicia automáticamente
docker ps
docker logs postgres-ha --tail 10

✅ **Resultado:** El contenedor se reinicia automáticamente gracias a `--restart always`. Los datos persisten porque están en el volumen.

---
## **Parte 6: Introducción a Alta Disponibilidad (Conceptos)**

La alta disponibilidad (HA) busca minimizar el tiempo de inactividad. Estrategias comunes:

1. **Réplicas de lectura**: Copias de la BD para distribuir consultas.
2. **Failover automático**: Si el primario falla, un secundario toma el control.
3. **Clustering**: Múltiples nodos trabajando juntos.

### **Ejemplo: Replicación en Docker (conceptual)**

Para configurar una réplica de PostgreSQL en Docker, necesitaríamos:
*   Dos contenedores en la misma red.
*   Configurar el primario con `wal_level = replica`.
*   Hacer un `pg_basebackup` desde el secundario.
*   Configurar `primary_conninfo` en el secundario.

Este ejercicio es más complejo y lo veremos en el NB2.

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica lo aprendido en Play with Docker.

### **Ejercicio 1: Levantar MySQL en Docker**

Repite los pasos de la Parte 2 pero con MySQL:
1. Descarga la imagen `mysql:8`.
2. Ejecuta un contenedor con las variables de entorno `MYSQL_ROOT_PASSWORD=root`, `MYSQL_DATABASE=prueba`.
3. Conéctate al contenedor usando `docker exec -it ... mysql -p`.
4. Crea una tabla e inserta datos.

In [ ]:
# Escribe aquí los comandos que usarías (no es código ejecutable en este notebook, pero puedes probarlos en PWD)

# 1. Descargar imagen
# docker pull mysql:8

# 2. Ejecutar contenedor
# docker run -d --name mysql1 -e MYSQL_ROOT_PASSWORD=root -e MYSQL_DATABASE=prueba -p 3306:3306 mysql:8

# 3. Conectarse
# docker exec -it mysql1 mysql -p

# 4. Comandos SQL dentro de MySQL:
# USE prueba;
# CREATE TABLE test (id INT AUTO_INCREMENT PRIMARY KEY, nombre VARCHAR(100));
# INSERT INTO test (nombre) VALUES ('Ejemplo1'), ('Ejemplo2');
# SELECT * FROM test;
# EXIT;

### **Ejercicio 2: Persistencia con bind mount**

En lugar de usar un volumen nombrado, usa un **bind mount** (mapear un directorio del host al contenedor).

1. Crea un directorio en el host: `mkdir ~/pgdata`.
2. Ejecuta PostgreSQL montando ese directorio: `-v ~/pgdata:/var/lib/postgresql/data`.
3. Crea datos, elimina el contenedor y crea uno nuevo con el mismo bind mount. Verifica que los datos persisten.

In [ ]:
# Comandos para el ejercicio 2

# 1. Crear directorio
# mkdir ~/pgdata

# 2. Ejecutar primer contenedor
# docker run -d --name postgres-bind -e POSTGRES_PASSWORD=admin123 -v ~/pgdata:/var/lib/postgresql/data -p 5436:5432 postgres:15

# 3. Crear datos
# docker exec -it postgres-bind psql -U postgres -c "CREATE DATABASE prueba;"
# docker exec -it postgres-bind psql -U postgres -d prueba -c "CREATE TABLE items (id SERIAL, nombre TEXT);"
# docker exec -it postgres-bind psql -U postgres -d prueba -c "INSERT INTO items (nombre) VALUES ('bind1'), ('bind2');"

# 4. Eliminar y recrear
# docker stop postgres-bind
# docker rm postgres-bind
# docker run -d --name postgres-bind-nuevo -e POSTGRES_PASSWORD=admin123 -v ~/pgdata:/var/lib/postgresql/data -p 5437:5432 postgres:15

# 5. Verificar datos
# docker exec -it postgres-bind-nuevo psql -U postgres -d prueba -c "SELECT * FROM items;"

### **Ejercicio 3: Red con dos aplicaciones**

Crea una red llamada `miapp`. Levanta:
1. Un contenedor PostgreSQL llamado `db`.
2. Un contenedor con **Adminer** (cliente web de BD) usando la imagen `adminer`.

Conéctate a Adminer desde el navegador (puerto 8080) y conéctate al servidor `db` con usuario `admin` y contraseña `admin123`.

*Pista: Adminer necesita el puerto 8080 mapeado: `-p 8080:8080`.*

In [ ]:
# Comandos para el ejercicio 3

# 1. Crear red
# docker network create miapp

# 2. Levantar PostgreSQL
# docker run -d --name db --network miapp -e POSTGRES_PASSWORD=admin123 -e POSTGRES_USER=admin -e POSTGRES_DB=testdb postgres:15

# 3. Levantar Adminer
# docker run -d --name adminer --network miapp -p 8080:8080 adminer

# 4. Abrir navegador en http://localhost:8080 y conectar:
#    Sistema: PostgreSQL
#    Servidor: db
#    Usuario: admin
#    Contraseña: admin123
#    Base de datos: testdb

### **Ejercicio 4: Reflexión sobre alta disponibilidad**

Responde en una celda Markdown:
1. ¿Qué ventajas ofrece ejecutar bases de datos en contenedores para lograr alta disponibilidad?
2. ¿Qué limitaciones tienen los contenedores en escenarios de alta disponibilidad en producción?
3. Investiga: ¿Qué es un `StatefulSet` en Kubernetes y por qué es importante para bases de datos?

**Tus respuestas:**

1. **Ventajas de contenedores para alta disponibilidad:**
   - Rápido aprovisionamiento y escalado.
   - Entornos consistentes en desarrollo y producción.
   - Fácil replicación mediante imágenes.
   - Aislamiento de recursos.
   - Integración con orquestadores (Kubernetes) que gestionan el ciclo de vida.

2. **Limitaciones:**
   - Los contenedores son efímeros por naturaleza, requieren volúmenes externos para persistencia.
   - La gestión de estado distribuido es compleja.
   - El failover automático requiere configuración adicional (no es nativo de Docker).
   - La latencia de red puede ser mayor que en despliegues bare-metal.

3. **StatefulSet en Kubernetes:**
   - Es un controlador de Kubernetes diseñado específicamente para aplicaciones con estado (como bases de datos).
   - Proporciona identidades estables (nombres de red persistentes) para cada pod.
   - Garantiza orden en el despliegue y escalado.
   - Gestiona volúmenes persistentes asociados a cada réplica.
   - Es importante para bases de datos porque mantiene la identidad y el almacenamiento incluso cuando los pods se recrean.

---
## **Conclusiones**

En esta sesión hemos:
1. **Aprendido** los fundamentos de Docker aplicados a bases de datos.
2. **Desplegado** PostgreSQL en contenedores con persistencia usando volúmenes.
3. **Conectado** múltiples contenedores mediante redes Docker.
4. **Simulado** fallos y verificado el reinicio automático.
5. **Introducido** conceptos de alta disponibilidad.

**Conceptos clave:**
*   Los contenedores facilitan el despliegue y la portabilidad.
*   Los volúmenes son esenciales para la persistencia de datos.
*   Las redes permiten la comunicación entre contenedores.
*   Políticas de reinicio (`--restart`) proporcionan resiliencia básica.

En el próximo notebook, exploraremos configuraciones más avanzadas: replicación, balanceo de carga y orquestación con Kubernetes.

In [ ]:
# Limpieza opcional (en PWD, las instancias se borran al cerrar)
# docker stop $(docker ps -aq)
# docker rm $(docker ps -aq)
# docker volume prune -f
# docker network prune -f
print("\nSesión completada. Puedes cerrar la ventana de Play with Docker.")